# Windows Repository Bridge — direct Colab vision server (Ollama)

This notebook runs a **prebuilt Ollama runtime** with Qwen3-VL vision support. It does not compile llama.cpp, build a CUDA server, or require a separate multimodal projector download.

1. Open this notebook using **Connect Colab** in the Windows app, select a GPU runtime (T4 is supported), and run the worker cell.
2. The cell starts the authenticated HTTPS endpoint and copies the temporary URL. Once the Windows app registers it, vision activates and the model begins loading automatically. No separate activation command, GitHub token, or Colab Secret is required.
3. The notebook checks the GPU, installs Ollama from its official prebuilt release if needed, and downloads **Qwen3-VL 4B Instruct (about 3.3 GB)** to Colab's local disk—not Google Drive. Once the model download is confirmed and the local Ollama API is healthy, the status becomes `ready`. Startup does not run image inference; the first real screenshot request may take longer while Ollama loads the model onto the GPU.
4. Watch `computer(action=status)`. After it reports `ready`, use `computer(action=screenshot)` or a batched `computer(action=sequence, observe=true)` to analyze a real Windows screenshot.

**Startup trade-off:** Colab's local disk avoids the slow Google Drive I/O and Drive authorization step, but it is temporary. A fresh Colab runtime must download the model again. The progress stage and elapsed time are visible through the status endpoint. Each activation attempt has an eight-minute overall setup budget; a timeout stops that attempt and includes diagnostic log tails. It will not automatically repeat the same failed download; call `computer(action=enable)` to deliberately retry.

The desktop app sends screenshots directly to this temporary authenticated HTTPS endpoint. The notebook does not read or write GitHub. It uses a random in-memory bearer key, exposes only health/activation/deactivation/screenshot-analysis routes, and verifies each screenshot SHA-256. Computer input is executed locally by the Windows app; Colab only receives screenshots for visual analysis. Use `computer(action=start_control)` to authorize a session for fast consecutive inputs and `computer(action=stop_control)` to release held keys/buttons. Deactivation leaves the Colab runtime and model running. Colab GPU access and session uptime are not guaranteed.

Treat the temporary endpoint URL as a password; do not share it. If automatic clipboard copy is blocked, click the notebook's endpoint copy button and paste the full URL into the Windows app's **Paste URL** control.

In [ ]:
# @title Start the direct Colab vision server
import base64
import hashlib
import hmac
import json
import os
import re
import secrets
import shutil
import subprocess
import threading
import time
from datetime import datetime, timezone
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path

import requests
from google.colab import output

MODEL_REPO = "ollama/qwen3-vl"
OLLAMA_MODEL = "qwen3-vl:4b-instruct"
OLLAMA_API = "http://127.0.0.1:11434"
OLLAMA_MODELS_DIR = Path("/content/ollama-models")
OLLAMA_LOG = Path("/content/ollama-serve.log")
DIRECT_API_PORT = 8765
DIRECT_PROTOCOL = "chatgpt-colab-vision-direct/v1"
RESULT_PROTOCOL = "chatgpt-colab-vision-result/v1"
MAX_SCREENSHOT_BYTES = 600 * 1024
POLL_SECONDS = 3
HEARTBEAT_SECONDS = 15
API_KEY = secrets.token_urlsafe(32)

# Make rerunning the cell replace the old tunnel/API threads, but keep a live loaded model.
_previous_api_server = globals().get("_direct_api_server")
_previous_tunnel_process = globals().get("_tunnel_process")
_previous_tunnel_log_handle = globals().get("_tunnel_log_handle")
_previous_model_server = globals().get("_server")
_previous_model_loaded = globals().get("_model_loaded", False)
_worker_generation = globals().get("_worker_generation", 0) + 1
if _previous_api_server is not None:
    try:
        _previous_api_server.shutdown()
        _previous_api_server.server_close()
    except Exception:
        pass
if _previous_tunnel_process is not None and _previous_tunnel_process.poll() is None:
    try:
        _previous_tunnel_process.terminate()
        _previous_tunnel_process.wait(timeout=8)
    except Exception:
        try:
            _previous_tunnel_process.kill()
        except Exception:
            pass
if _previous_tunnel_log_handle is not None:
    try:
        _previous_tunnel_log_handle.close()
    except Exception:
        pass
# Reuse an Ollama process across cell reruns; stop an older llama.cpp server to free VRAM.
_previous_args = str(getattr(_previous_model_server, "args", "")).lower()
if _previous_model_server is not None and _previous_model_server.poll() is None and "ollama" not in _previous_args:
    try:
        _previous_model_server.terminate()
        _previous_model_server.wait(timeout=8)
    except Exception:
        try:
            _previous_model_server.kill()
        except Exception:
            pass
_is_previous_ollama = (
    _previous_model_server is not None
    and _previous_model_server.poll() is None
    and "ollama" in _previous_args
)
_server = _previous_model_server if _is_previous_ollama else None
_model_loaded = bool(_server is not None and _previous_model_loaded)
_previous_ollama_log = globals().get("_ollama_log_handle")
_ollama_log_handle = _previous_ollama_log if _is_previous_ollama else None
_direct_api_server = None
_tunnel_process = None
_tunnel_log_handle = None
_enabled = False
_gpu = None
_vram_mib = None
_active_request = None
_last_completed = None
_runtime_lock = threading.RLock()
_inference_lock = threading.Lock()
_last_setup_attempt = 0.0
_setup_deadline = None
_setup_blocked = False
SETUP_BUDGET_SECONDS = 480  # hard cap: eight minutes per explicit activation attempt

# After the notebook copies the public endpoint, the Windows app discovers it from the clipboard and confirms the HTTPS connection.
# The notebook never asks the browser to access localhost.
_desktop_registration_event = threading.Event()
_desktop_registration_at = None

_stage_started_monotonic = time.monotonic()
_stage_started_at = datetime.now(timezone.utc).isoformat()
_last_stage_notice = None

_runtime_state = {
    "status": "inactive",
    "stage": "server_start",
    "message": "Direct Colab service is starting. Vision automatically activates after authenticated desktop registration.",
    "model_loaded": False,
    "gpu": None,
    "vram_mib": None,
    "active_request_id": None,
    "last_completed_request_id": None,
    "updated_at": None
}

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def publish_runtime(status, stage=None, message=None, **extra):
    global _runtime_state, _stage_started_monotonic, _stage_started_at, _last_stage_notice
    with _runtime_lock:
        payload = dict(_runtime_state)
        next_stage = stage if stage is not None else payload.get("stage")
        next_message = (message if message is not None else payload.get("message") or "")[:1000]
        if next_stage != payload.get("stage"):
            _stage_started_monotonic = time.monotonic()
            _stage_started_at = utc_now()
        notice = (str(status), str(next_stage), next_message)
        if notice != _last_stage_notice:
            print("[vision] stage=" + str(next_stage) + " status=" + str(status) +
                  " elapsed=0s — " + next_message, flush=True)
            _last_stage_notice = notice
        payload.update({
            "protocol": DIRECT_PROTOCOL,
            "status": status,
            "enabled": bool(_enabled),
            "heartbeat_at": utc_now(),
            "updated_at": utc_now(),
            "worker": "colab-direct-vision/v1",
            "model_repository": MODEL_REPO,
            "model": OLLAMA_MODEL,
            "projector": None,
            "model_loaded": bool(_model_loaded),
            "gpu": _gpu,
            "vram_mib": _vram_mib,
            "active_request_id": _active_request,
            "last_completed_request_id": _last_completed,
            "stage": next_stage,
            "stage_started_at": _stage_started_at,
            "stage_elapsed_seconds": max(0, int(time.monotonic() - _stage_started_monotonic)),
            "message": next_message
        })
        payload.update(extra)
        _runtime_state = payload
        return dict(payload)

def check_gpu():
    if not shutil.which("nvidia-smi"):
        return None
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader,nounits"],
        check=True, capture_output=True, text=True, timeout=15
    )
    line = next((line.strip() for line in result.stdout.splitlines() if line.strip()), "")
    columns = [column.strip() for column in line.split(",")]
    if len(columns) < 3:
        raise RuntimeError("Could not parse nvidia-smi GPU/VRAM output.")
    return columns[0], int(columns[1]), int(columns[2])

def bounded_setup_timeout(maximum):
    """Return a per-step timeout capped by the current overall setup deadline."""
    if _setup_deadline is None:
        return maximum
    remaining = int(_setup_deadline - time.monotonic())
    if remaining <= 0:
        raise TimeoutError(
            f"Vision setup exceeded its {SETUP_BUDGET_SECONDS}-second time budget. "
            "Use computer(action=status) to inspect the error and logs, then computer(action=enable) to retry."
        )
    return min(maximum, remaining)


def tail_log(path, limit=1800):
    try:
        return Path(path).read_text(encoding="utf-8", errors="replace")[-limit:]
    except Exception:
        return "(log unavailable)"


def run_setup_command(command, *, stage, message, timeout, env, log_path):
    """Run a bounded installer/pull command while keeping setup status observable."""
    started = time.monotonic()
    Path(log_path).parent.mkdir(parents=True, exist_ok=True)
    with open(log_path, "a", encoding="utf-8") as log_handle:
        process = subprocess.Popen(command, stdout=log_handle, stderr=subprocess.STDOUT,
                                   env=env, start_new_session=True)
        last_notice = 0.0
        while process.poll() is None:
            elapsed = time.monotonic() - started
            if elapsed >= timeout:
                process.terminate()
                try:
                    process.wait(timeout=8)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait(timeout=3)
                raise RuntimeError(
                    f"{message} timed out after {int(elapsed)} seconds. "
                    f"Recent log: {tail_log(log_path, 1200)}"
                )
            if elapsed - last_notice >= 15:
                last_notice = elapsed
                lines = tail_log(log_path, 350).strip().splitlines()
                detail = lines[-1][-220:] if lines else "Still running."
                publish_runtime("starting", stage=stage,
                                message=f"{message} Elapsed {int(elapsed)} seconds. {detail}")
            time.sleep(2)
        if process.returncode != 0:
            raise RuntimeError(
                f"{message} failed with exit code {process.returncode}. "
                f"Recent log: {tail_log(log_path, 1600)}"
            )
    return int(time.monotonic() - started)


def ollama_model_is_downloaded():
    try:
        response = requests.get(OLLAMA_API + "/api/tags", timeout=5)
        if not response.ok:
            return False
        models = response.json().get("models", [])
        return any(str(m.get("name", m.get("model", ""))) == OLLAMA_MODEL
                   for m in models if isinstance(m, dict))
    except Exception:
        return False


def ensure_ollama_binary(env):
    binary = shutil.which("ollama")
    if binary:
        version = subprocess.run([binary, "--version"], capture_output=True, text=True,
                                 timeout=15, env=env)
        if version.returncode == 0:
            print("[vision] Reusing preinstalled Ollama:", (version.stdout or version.stderr).strip()[:200])
            return binary

    publish_runtime("starting", stage="install_ollama",
                    message="Downloading the official prebuilt Ollama runtime. No source compilation is used.")
    installer_path = Path("/content/ollama-install.sh")
    response = requests.get("https://ollama.com/install.sh", timeout=bounded_setup_timeout(60))
    response.raise_for_status()
    installer = response.content
    if len(installer) < 1000 or b"download_and_extract" not in installer or b"ollama-linux" not in installer:
        raise RuntimeError("The official Ollama installer response was unexpectedly short or invalid.")
    installer_path.write_bytes(installer)

    # Current Linux releases can use zstd-compressed archives. Install only this small
    # decompressor if missing; the Ollama server and GPU runtime remain prebuilt.
    if not shutil.which("zstd"):
        run_setup_command(
            ["bash", "-lc", "apt-get update -qq && apt-get install -y -qq zstd"],
            stage="install_dependencies",
            message="Preparing the decompressor for the official prebuilt package.",
            timeout=bounded_setup_timeout(150), env=env, log_path=Path("/content/ollama-apt.log")
        )
        if not shutil.which("zstd"):
            raise RuntimeError("zstd installation completed but the executable is still missing.")

    install_env = env.copy()
    install_env["OLLAMA_NO_START"] = "1"
    run_setup_command(
        ["sh", str(installer_path)], stage="install_ollama",
        message="Installing the official prebuilt Ollama binary and runtime libraries.",
        timeout=bounded_setup_timeout(180), env=install_env, log_path=Path("/content/ollama-install.log")
    )
    binary = shutil.which("ollama") or ("/usr/local/bin/ollama" if Path("/usr/local/bin/ollama").is_file() else None)
    if not binary:
        raise RuntimeError("The official installer completed but the ollama executable could not be found.")
    version = subprocess.run([binary, "--version"], capture_output=True, text=True,
                             timeout=15, env=env)
    version_text = (version.stdout or version.stderr).strip()
    if version.returncode != 0:
        raise RuntimeError("The installed Ollama binary failed its version check: " + version_text[:400])
    print("[vision] Installed prebuilt Ollama:", version_text[:200])
    return binary


def ensure_server():
    global _server, _model_loaded, _gpu, _vram_mib, _last_setup_attempt, _ollama_log_handle, _setup_deadline
    if _server is not None and _server.poll() is None:
        try:
            if requests.get(OLLAMA_API + "/api/version", timeout=2).ok and ollama_model_is_downloaded():
                _model_loaded = True
                publish_runtime("ready", stage="ready",
                                message="Ollama is running Qwen3-VL 4B; the vision model remains loaded.")
                return True
        except requests.RequestException:
            pass

    if time.monotonic() - _last_setup_attempt < 45:
        return False
    _last_setup_attempt = time.monotonic()
    _setup_deadline = time.monotonic() + SETUP_BUDGET_SECONDS
    _model_loaded = False
    publish_runtime("starting", stage="detect_gpu",
                    message="Checking the Colab GPU before installing or downloading the vision model.")
    gpu = check_gpu()
    if not gpu:
        publish_runtime("gpu_unavailable", stage="detect_gpu",
                        message="No NVIDIA GPU is attached to this Colab runtime. Select a GPU runtime if one is available.")
        return False
    _gpu, total_mib, free_mib = gpu
    _vram_mib = total_mib
    if total_mib < 8000:
        publish_runtime("gpu_unavailable", stage="check_vram",
                        message=f"Detected {_gpu} with {total_mib} MiB VRAM. Qwen3-VL 4B needs a GPU with about 8 GB or more VRAM.")
        return False

    # Use Colab's fast, temporary disk rather than Google Drive. The model must be
    # redownloaded after a fresh runtime, but there is no Drive mount or Drive I/O.
    OLLAMA_MODELS_DIR.mkdir(parents=True, exist_ok=True)
    server_env = os.environ.copy()
    server_env.update({
        "OLLAMA_HOST": "127.0.0.1:11434",
        "OLLAMA_MODELS": str(OLLAMA_MODELS_DIR),
        "OLLAMA_KEEP_ALIVE": "-1",
        "OLLAMA_NUM_PARALLEL": "1",
        "OLLAMA_MAX_LOADED_MODELS": "1"
    })
    binary = ensure_ollama_binary(server_env)

    try:
        version_response = requests.get(OLLAMA_API + "/api/version", timeout=2)
    except requests.RequestException:
        version_response = None
    if version_response is None or not version_response.ok:
        if _ollama_log_handle is None or getattr(_ollama_log_handle, "closed", True):
            _ollama_log_handle = open(OLLAMA_LOG, "a", encoding="utf-8")
        _server = subprocess.Popen([binary, "serve"], stdout=_ollama_log_handle,
                                   stderr=subprocess.STDOUT, start_new_session=True, env=server_env)
        publish_runtime("starting", stage="start_ollama",
                        message="The prebuilt Ollama server is starting on localhost; no source build is running.")
        deadline = time.monotonic() + min(45, bounded_setup_timeout(45))
        last_error = "Ollama has not responded yet."
        while time.monotonic() < deadline:
            if _server.poll() is not None:
                raise RuntimeError("Ollama exited before its API became healthy. Recent log: " + tail_log(OLLAMA_LOG, 1500))
            try:
                version_response = requests.get(OLLAMA_API + "/api/version", timeout=2)
                if version_response.ok and isinstance(version_response.json().get("version"), str):
                    break
                last_error = "Ollama's version endpoint returned an invalid response."
            except Exception as exc:
                last_error = type(exc).__name__ + ": " + str(exc)[:250]
            time.sleep(2)
        else:
            raise RuntimeError("Ollama did not start within 60 seconds: " + last_error + ". Recent log: " + tail_log(OLLAMA_LOG, 1200))

    if not ollama_model_is_downloaded():
        publish_runtime("starting", stage="download_model",
                        message="Downloading Qwen3-VL 4B (about 3.3 GB) to Colab local storage. This is a model download, not a build.")
        run_setup_command(
            [binary, "pull", OLLAMA_MODEL], stage="download_model",
            message="Downloading the Qwen3-VL 4B vision model to Colab local storage.",
            timeout=bounded_setup_timeout(420), env=server_env, log_path=Path("/content/ollama-pull.log")
        )
    if not ollama_model_is_downloaded():
        raise RuntimeError("Ollama pull finished but qwen3-vl:4b is not listed by the local API.")

    # Readiness means Ollama is reachable and the model weights are available.
    # Do not send a synthetic image at startup; the first real screenshot request
    # performs the first image inference and may need extra time to load the model.
    _model_loaded = True
    publish_runtime("ready", stage="ready",
                    message="Ollama is healthy and Qwen3-VL 4B is downloaded. No startup image inference was run; the first real screenshot may take longer while the model warms up.",
                    model_runtime="Ollama", model_loaded=True)
    return True



class DirectVisionHandler(BaseHTTPRequestHandler):
    server_version = "BridgeVision/1"

    def log_message(self, fmt, *args):
        # Do not log bearer credentials, image payloads, or user prompts.
        return

    def _authorized(self):
        value = self.headers.get("Authorization", "")
        return hmac.compare_digest(value, "Bearer " + API_KEY)

    def _route_path(self):
        # Public requests must include the random endpoint key in the URL path.
        # This makes the copied endpoint itself a capability while retaining the
        # Bearer header check as a second layer. Cloudflared forwards this path.
        raw_path = self.path.split("?", 1)[0]
        prefix = "/" + API_KEY
        if not raw_path.startswith(prefix + "/"):
            return None
        return raw_path[len(prefix):]

    def _send(self, status, payload):
        data = json.dumps(payload, ensure_ascii=False).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Cache-Control", "no-store")
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        if not self._authorized():
            self._send(401, {"error": "Unauthorized"})
            return
        route = self._route_path()
        if route != "/health":
            self._send(404, {"error": "Not found"})
            return
        with _runtime_lock:
            payload = dict(_runtime_state)
            payload["stage_started_at"] = _stage_started_at
            payload["stage_elapsed_seconds"] = max(0, int(time.monotonic() - _stage_started_monotonic))
        payload.update({
            "protocol": DIRECT_PROTOCOL,
            "enabled": bool(_enabled),
            "model_loaded": bool(_model_loaded),
            "gpu": _gpu,
            "vram_mib": _vram_mib,
            "active_request_id": _active_request,
            "last_completed_request_id": _last_completed,
            "heartbeat_at": utc_now(),
            "updated_at": utc_now()
        })
        self._send(200, payload)

    def do_POST(self):
        global _enabled, _active_request, _last_completed, _setup_blocked, _last_setup_attempt
        if not self._authorized():
            self._send(401, {"error": "Unauthorized"})
            return
        route = self._route_path()
        if route is None:
            self._send(404, {"error": "Not found"})
            return
        try:
            length = int(self.headers.get("Content-Length", "0"))
        except ValueError:
            self._send(400, {"error": "Invalid Content-Length"})
            return
        if length < 1 or length > 2_000_000:
            self._send(413, {"error": "Request body must be between 1 byte and 2 MB"})
            return
        try:
            body = json.loads(self.rfile.read(length).decode("utf-8"))
        except Exception:
            self._send(400, {"error": "Malformed JSON"})
            return
        if not isinstance(body, dict) or body.get("protocol") != DIRECT_PROTOCOL:
            self._send(400, {"error": "Protocol mismatch"})
            return
        if route == "/registration/ack":
            global _desktop_registration_at
            _desktop_registration_at = utc_now()
            # Registration itself is the activation request; do not depend on a second
            # desktop request arriving after the acknowledgement.
            _enabled = True
            _setup_blocked = False
            _last_setup_attempt = 0.0
            publish_runtime(
                "starting",
                stage="activation",
                message="The Windows app verified the authenticated HTTPS tunnel. Vision is active; model setup is starting in the background.",
                desktop_registered=True,
                desktop_registered_at=_desktop_registration_at
            )
            self._send(200, {
                "protocol": DIRECT_PROTOCOL,
                "accepted": True,
                "desktop_registered": True,
                "vision_activated": True,
                "desktop_registered_at": _desktop_registration_at
            })
            _desktop_registration_event.set()
            return
        if route == "/activate":
            _enabled = True
            _setup_blocked = False
            _last_setup_attempt = 0.0
            publish_runtime("starting", stage="activation", message="Vision activation received. Model setup is starting.")
            self._send(202, {"protocol": DIRECT_PROTOCOL, "accepted": True, "enabled": True})
            return
        if route == "/deactivate":
            _enabled = False
            publish_runtime("inactive", stage="idle", message="Vision is disabled by an explicit user request; the runtime and any loaded model remain running.")
            self._send(200, {"protocol": DIRECT_PROTOCOL, "accepted": True, "enabled": False, "runtime_kept_alive": True})
            return
        if route != "/analyze":
            self._send(404, {"error": "Not found"})
            return
        if not _enabled:
            self._send(409, {"error": "Vision is inactive; call computer(action=enable) first."})
            return
        if not _model_loaded:
            self._send(409, {"error": "Vision model is not loaded yet; check computer(action=status) and retry the same request ID."})
            return
        with _inference_lock:
            try:
                job_id = body.get("request_id")
                if not isinstance(job_id, str) or not re.fullmatch(r"computer-[a-f0-9]{24}", job_id):
                    self._send(400, {"error": "Malformed request ID"})
                    return
                expected_hash = body.get("sha256")
                image_b64 = body.get("image_base64")
                if not isinstance(expected_hash, str) or not re.fullmatch(r"[a-f0-9]{64}", expected_hash):
                    self._send(400, {"error": "Malformed SHA-256"})
                    return
                if not isinstance(image_b64, str):
                    self._send(400, {"error": "Image payload is missing"})
                    return
                image_bytes = base64.b64decode(image_b64, validate=True)
                if not image_bytes or len(image_bytes) > MAX_SCREENSHOT_BYTES:
                    self._send(413, {"error": "Screenshot is empty or larger than the 600 KiB limit"})
                    return
                actual_hash = hashlib.sha256(image_bytes).hexdigest()
                if not hmac.compare_digest(actual_hash, expected_hash):
                    self._send(400, {"error": "Screenshot SHA-256 did not match the submitted value"})
                    return
                _active_request = job_id
                publish_runtime("processing", stage="inference", message="Analyzing one screenshot received from the desktop app.")
                started = time.monotonic()
                request_payload = {
                    "model": OLLAMA_MODEL,
                    "stream": False,
                    "think": False,
                    "keep_alive": -1,
                    "messages": [
                        {
                            "role": "system",
                            "content": (
                                "You are a fast computer-use visual observer, not a general reasoning agent. "
                                "Describe only what the image supports and mark uncertainty. Return concise structured output with: "
                                "visible app/game state; important targets and controls with approximate bounding boxes and center points "
                                "in screenshot pixel coordinates (origin at top-left); important readable text; cursor/player/HUD state; "
                                "and any visible errors. Do not invent exact coordinates if unclear. Avoid long prose and do not infer unseen state."
                            )
                        },
                        {
                            "role": "user",
                            "content": str(body.get("prompt") or "")[:2000],
                            "images": [image_b64]
                        }
                    ],
                    "options": {"temperature": 0, "num_predict": 1200, "num_ctx": 4096}
                }
                try:
                    response = requests.post(OLLAMA_API + "/api/chat", json=request_payload, timeout=(10, 300))
                    if not response.ok:
                        raise RuntimeError("Ollama HTTP " + str(response.status_code) + ": " + response.text[:600])
                    output = response.json()
                    if not isinstance(output, dict):
                        raise RuntimeError(
                            "Ollama /api/chat returned a non-object JSON payload "
                            f"({type(output).__name__})."
                        )
                    message = output.get("message", {})
                    if not isinstance(message, dict):
                        message = {}
                    report = message.get("content")
                    if not isinstance(report, str) or not report.strip():
                        # Never return the separate reasoning field as if it were a user-facing answer.
                        # Include only structural diagnostics (not model reasoning or screenshot).
                        diagnostics = {
                            "model": output.get("model"),
                            "done": output.get("done"),
                            "done_reason": output.get("done_reason"),
                            "prompt_eval_count": output.get("prompt_eval_count"),
                            "eval_count": output.get("eval_count"),
                            "message_keys": sorted(message.keys()),
                            "content_chars": len(message.get("content")) if isinstance(message.get("content"), str) else None,
                            "thinking_chars": len(message.get("thinking")) if isinstance(message.get("thinking"), str) else None,
                        }
                        raise RuntimeError(
                            "Ollama /api/chat returned no final message.content; diagnostics="
                            + json.dumps(diagnostics, sort_keys=True)
                        )
                    report = report.strip()
                    if len(report) > 30000:
                        report = report[:30000] + "\n\n[Report truncated at the safety limit.]"
                    _last_completed = job_id
                    result = {
                        "protocol": RESULT_PROTOCOL,
                        "request_id": job_id,
                        "status": "completed",
                        "completed_at": utc_now(),
                        "model_repository": MODEL_REPO,
                        "model": OLLAMA_MODEL,
                        "input_sha256": actual_hash,
                        "inference_ms": int((time.monotonic() - started) * 1000),
                        "report": report
                    }
                    self._send(200, result)
                except Exception as exc:
                    self._send(200, {
                        "protocol": RESULT_PROTOCOL,
                        "request_id": job_id,
                        "status": "failed",
                        "completed_at": utc_now(),
                        "input_sha256": actual_hash,
                        "error": type(exc).__name__ + ": " + str(exc)[:1200]
                    })
                finally:
                    _active_request = None
                    publish_runtime("ready" if _model_loaded else "error", stage="ready" if _model_loaded else "inference_error",
                                    message="Ollama vision model is ready for another request." if _model_loaded else "Ollama vision health was lost during inference.")
            except Exception as exc:
                self._send(400, {"error": type(exc).__name__ + ": " + str(exc)[:800]})

def api_server_loop():
    global _direct_api_server
    server = ThreadingHTTPServer(("127.0.0.1", DIRECT_API_PORT), DirectVisionHandler)
    server.daemon_threads = True
    _direct_api_server = server
    print("Authenticated vision API listening on localhost port", DIRECT_API_PORT)
    server.serve_forever()

def worker_loop(generation):
    global _model_loaded, _setup_blocked
    while generation == _worker_generation:
        try:
            if _enabled:
                if _setup_blocked:
                    time.sleep(POLL_SECONDS)
                    continue
                if _model_loaded and not ollama_model_is_downloaded():
                    _model_loaded = False
                if not _model_loaded:
                    try:
                        ensure_server()
                    except Exception as exc:
                        _setup_blocked = True
                        publish_runtime("error", stage="setup_error", message=type(exc).__name__ + ": " + str(exc))
                else:
                    publish_runtime("ready", stage="ready", message="Vision is enabled and the model is ready.")
            else:
                publish_runtime("inactive", stage="idle", message="Direct Colab server is connected but waiting for the Windows app to auto-activate vision after registration.")
        except Exception as exc:
            _setup_blocked = True
            publish_runtime("error", stage="setup_error", message=type(exc).__name__ + ": " + str(exc))
        time.sleep(POLL_SECONDS)

def ensure_cloudflared():
    binary = shutil.which("cloudflared")
    if binary:
        print("[tunnel] Reusing cloudflared binary already on PATH.")
        return binary
    binary_path = Path("/content/cloudflared")
    if not binary_path.exists():
        url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
        print("[tunnel] Downloading cloudflared binary (network timeout: 90 seconds).")
        started = time.monotonic()
        try:
            response = requests.get(url, timeout=90)
            response.raise_for_status()
        except Exception as exc:
            raise RuntimeError("cloudflared binary download failed after " +
                               str(round(time.monotonic() - started, 1)) + " seconds: " +
                               type(exc).__name__ + ": " + str(exc)[:500]) from exc
        binary_path.write_bytes(response.content)
        binary_path.chmod(0o755)
        print("[tunnel] cloudflared download completed in " +
              str(round(time.monotonic() - started, 1)) + " seconds.")
    if not os.access(binary_path, os.X_OK):
        binary_path.chmod(0o755)
    return str(binary_path)


def start_cloudflared():
    global _tunnel_process, _tunnel_log_handle
    binary = ensure_cloudflared()
    log_path = Path("/content/bridge-cloudflared.log")
    _tunnel_log_handle = open(log_path, "w", encoding="utf-8")
    tunnel_started = time.monotonic()
    _tunnel_process = subprocess.Popen(
        [binary, "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:" + str(DIRECT_API_PORT)],
        stdout=_tunnel_log_handle, stderr=subprocess.STDOUT, start_new_session=True
    )
    print("[tunnel] cloudflared started; waiting up to 90 seconds for an HTTPS URL.")
    deadline = time.monotonic() + 90
    pattern = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
    while time.monotonic() < deadline:
        if _tunnel_process.poll() is not None:
            tail = log_path.read_text(encoding="utf-8", errors="replace")[-1600:]
            raise RuntimeError("cloudflared exited after " +
                               str(round(time.monotonic() - tunnel_started, 1)) +
                               " seconds before creating its HTTPS tunnel: " + tail)
        text = log_path.read_text(encoding="utf-8", errors="replace")
        match = pattern.search(text)
        if match:
            print("[tunnel] HTTPS URL created in " +
                  str(round(time.monotonic() - tunnel_started, 1)) + " seconds.")
            return match.group(0)
        time.sleep(1)
    raise RuntimeError("Timed out after " + str(round(time.monotonic() - tunnel_started, 1)) +
                       " seconds waiting for a Cloudflare Quick Tunnel URL. Check /content/bridge-cloudflared.log.")

def verify_local_api_health():
    """Verify the authenticated API without touching the GPU or loading the model."""
    url = "http://127.0.0.1:" + str(DIRECT_API_PORT) + "/" + API_KEY + "/health"
    deadline = time.monotonic() + 8
    last_error = "No response yet."
    while time.monotonic() < deadline:
        try:
            response = requests.get(url, headers={"Authorization": "Bearer " + API_KEY}, timeout=2)
            if response.status_code == 200:
                payload = response.json()
                if payload.get("protocol") == DIRECT_PROTOCOL and payload.get("enabled") is False and isinstance(payload.get("model_loaded"), bool):
                    return
                last_error = "Health payload protocol or enabled/model-readiness state was unexpected."
            else:
                last_error = "HTTP " + str(response.status_code)
        except Exception as exc:
            last_error = type(exc).__name__ + ": " + str(exc)[:300]
        time.sleep(0.4)
    raise RuntimeError("Local authenticated API health check failed before tunnel setup: " + last_error)


def stop_connection_bootstrap():
    """Close the API and tunnel on failure without terminating a previously cached model process."""
    global _direct_api_server, _tunnel_process, _tunnel_log_handle
    server, _direct_api_server = _direct_api_server, None
    if server is not None:
        try:
            server.shutdown()
        except Exception:
            pass
        try:
            server.server_close()
        except Exception:
            pass
    tunnel, _tunnel_process = _tunnel_process, None
    if tunnel is not None and tunnel.poll() is None:
        try:
            tunnel.terminate()
            tunnel.wait(timeout=5)
        except Exception:
            try:
                tunnel.kill()
                tunnel.wait(timeout=3)
            except Exception:
                pass
    handle, _tunnel_log_handle = _tunnel_log_handle, None
    if handle is not None:
        try:
            handle.close()
        except Exception:
            pass


def display_public_endpoint(tunnel_url):
    """Display and copy the single public endpoint the desktop is watching for."""
    endpoint_url = tunnel_url.rstrip("/") + "/" + API_KEY
    # The URL includes the high-entropy path token; never emit the raw key separately.
    print("PUBLIC_VISION_ENDPOINT=" + endpoint_url, flush=True)
    js = r"""
(async () => {
  const endpoint = __ENDPOINT__;
  const panelId = "windows-bridge-colab-endpoint";
  const oldPanel = document.getElementById(panelId);
  if (oldPanel) oldPanel.remove();

  const panel = document.createElement("section");
  panel.id = panelId;
  panel.setAttribute("role", "status");
  panel.style.cssText = "position:fixed;right:16px;bottom:16px;z-index:2147483647;width:360px;max-width:calc(100vw - 32px);box-sizing:border-box;padding:14px;background:#fff;color:#202124;border:1px solid #9aa0a6;border-radius:10px;box-shadow:0 4px 20px rgba(0,0,0,.24);font:14px/1.4 Arial,sans-serif;";
  const title = document.createElement("div");
  title.textContent = "Windows Repository Bridge";
  title.style.cssText = "font-weight:700;font-size:15px;margin-bottom:6px;";
  const detail = document.createElement("div");
  detail.style.cssText = "overflow-wrap:anywhere;margin-bottom:10px;";
  const endpointLabel = document.createElement("textarea");
  endpointLabel.value = __ENDPOINT__;
  endpointLabel.readOnly = true;
  endpointLabel.setAttribute("aria-label", "Temporary Colab vision endpoint URL");
  endpointLabel.spellcheck = false;
  endpointLabel.style.cssText = "display:block;width:100%;height:62px;box-sizing:border-box;resize:vertical;overflow:auto;overflow-wrap:anywhere;word-break:break-all;font-size:11px;margin:8px 0;padding:6px;";
  const button = document.createElement("button");
  button.type = "button";
  button.textContent = "Copy endpoint for Windows Bridge";
  button.style.cssText = "padding:8px 12px;border:0;border-radius:6px;background:#1a73e8;color:#fff;font-weight:600;cursor:pointer;";
  panel.append(title, detail, endpointLabel, button);
  document.body.appendChild(panel);

  const setStatus = (message, failed=false) => {
    detail.textContent = String(message || "").slice(0,1000);
    detail.style.color = failed ? "#b3261e" : "#202124";
  };
  const copyEndpoint = async (userInitiated = false) => {
    try {
      await navigator.clipboard.writeText(__ENDPOINT__);
      setStatus("Public endpoint copied. Windows Repository Bridge will connect, automatically activate vision, and begin model loading.");
      button.textContent = "Endpoint copied";
      button.disabled = true;
      return true;
    } catch (error) {
      // Legacy copy is only attempted inside the notebook's user-click fallback.
      try {
        const temporary = document.createElement("textarea");
        temporary.value = __ENDPOINT__;
        temporary.setAttribute("readonly", "");
        temporary.style.cssText = "position:fixed;left:-10000px;top:0;";
        document.body.appendChild(temporary);
        temporary.focus();
        temporary.select();
        const copied = document.execCommand("copy");
        temporary.remove();
        if (copied) {
          setStatus("Public endpoint copied. Windows Repository Bridge is watching for it and will connect automatically. Vision remains disabled.");
          button.textContent = "Endpoint copied";
          button.disabled = true;
          return true;
        }
      } catch (_) {}
      if (userInitiated) {
        endpointLabel.focus();
        endpointLabel.select();
      }
      setStatus(userInitiated
        ? "Clipboard copy was blocked. The endpoint is selected; press Ctrl+C, then click Paste URL in Windows Repository Bridge."
        : "Automatic clipboard copy was blocked. Click Select endpoint (Ctrl+C), then press Ctrl+C and paste the URL into Windows Repository Bridge using Paste URL.", true);
      button.disabled = false;
      button.textContent = "Select endpoint (Ctrl+C)";
      return false;
    }
  };
  button.addEventListener("click", () => { void copyEndpoint(true); });
  // Attempt automatic copy as soon as the tunnel exists. The app ignores whatever
  // was on the clipboard before Connect Colab was clicked and watches for this URL.
  let copied = false;
  try {
    copied = await copyEndpoint();
  } catch (_) {}
  return {installed:true, endpointVisible:true, automaticClipboardCopy:copied, transport:"clipboard-detected-public-https"};
})()
""".replace("__ENDPOINT__", json.dumps(endpoint_url))
    try:
        ui_state = output.eval_js(js)
    except Exception as exc:
        stop_connection_bootstrap()
        raise RuntimeError(
            "Could not display the public Colab endpoint: " +
            type(exc).__name__ + ": " + str(exc)[:800] +
            ". No model was loaded. Rerun the cell to create a fresh endpoint."
        ) from exc
    print("[notebook] Public HTTPS endpoint displayed. Windows Repository Bridge will connect when it detects this copied URL.")
    print("[notebook] If automatic clipboard copy was blocked, click 'Copy endpoint for Windows Bridge' in the output panel.")


print("[1/3] Starting the lightweight authenticated Colab API. The model remains unloaded.")
publish_runtime("starting", stage="api_server", message="Starting the lightweight authenticated vision API. The model remains unloaded.")
threading.Thread(target=api_server_loop, name="bridge-direct-api", daemon=True).start()
time.sleep(1)
try:
    verify_local_api_health()
except Exception:
    stop_connection_bootstrap()
    raise
print("[1/3] Local API reachability and bearer authentication passed.")

print("[2/3] Creating the temporary HTTPS tunnel (maximum 90 seconds).")
try:
    tunnel_url = start_cloudflared()
except Exception:
    stop_connection_bootstrap()
    raise
print("[2/3] Tunnel running. Publishing the endpoint for automatic Windows clipboard detection; the app will activate vision after registration.")
display_public_endpoint(tunnel_url)

print("[3/3] Waiting up to 80 seconds for the Windows app's authenticated server-to-server acknowledgement.")
if not _desktop_registration_event.wait(timeout=90):
    stop_connection_bootstrap()
    raise RuntimeError(
        "The Windows app did not detect and confirm the public HTTPS endpoint within 90 seconds. "
        "Check that the endpoint was copied to the Windows clipboard; if needed, click 'Copy endpoint for Windows Bridge' in the notebook panel. "
        "The Windows app must have been told to Connect Colab before the cell generated this URL. "
        "This is a connection-stage failure, not a GPU/model-loading failure; the tunnel was stopped and no model was loaded."
    )

print("[3/3] Direct authenticated HTTPS connection registered successfully. The Windows app has automatically activated vision; the model is warming up.")
print("Use computer(action=status) to check model warm-up. Once status is ready, use computer(action=screenshot) or a batched computer(action=sequence, observe=true) for visual feedback.")
print("The connection check did not download or activate the model. Keep this cell running while using vision.")
threading.Thread(target=worker_loop, args=(_worker_generation,), name="bridge-vision-worker", daemon=True).start()
while True:
    time.sleep(60)
